# Archived student research: superseded

Authors: Paul Trassaert, Lebel, Lesage, Dherissart.

This is a source archive, not the runnable reference. Original code cells are retained below; saved outputs, assignment prose and environment installation cells were removed. Undefined meta-labelling variables and other issues remain deliberately visible. See `../AUDIT.md`. Do not interpret historical claims inside code comments as verified. Use `../run.py` for the corrected implementation.

## Original cell 7 (zero-based index)

In [ ]:
# Prices
print("Loading prices data...")
url = 'https://drive.google.com/uc?id=1P_5ykYLd5521QUdCxC_cMytdJ3PqESTw'
prices = pd.read_csv(url, parse_dates=True, index_col=0)
print("✓ Loaded prices")

## Original cell 8 (zero-based index)

In [ ]:
prices.head(10)

## Original cell 14 (zero-based index)

In [ ]:
train_data = prices.loc['2018-01-01':'2020-12-31'].copy()
test_data = prices.loc['2021-01-01':'2021-12-31'].copy()

print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")

TRAIN_START = '2018-01-01'
TRAIN_END   = '2020-12-31'
TEST_START  = '2021-01-01'
TEST_END    = '2021-12-31'

## Original cell 15 (zero-based index)

In [ ]:
warnings.filterwarnings('ignore')

def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute a comprehensive, institutional-grade feature matrix from OHLCV data.
    Incorporates Fractional Differencing, Advanced Volatility, and Latent Variable Models.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with columns: open, high, low, close, volume (Daily frequency assumed)

    Returns
    -------
    pd.DataFrame
        Feature matrix (NaN rows removed)
    """
    print("Computing quantitative features...")
    feat = pd.DataFrame(index=df.index)
    close = df['close']
    high  = df['high']
    low   = df['low']
    opn   = df['open']
    vol   = df['volume']
    log_ret = np.log(close / close.shift(1))
    feat['ret_1h'] = log_ret
    for lag in [12, 24, 72, 168]:
        feat[f'ret_cumul_{lag}h'] = log_ret.rolling(lag).sum()

    # MOMENTUM & CLASSICAL TA
    feat['rsi_72h']    = ta.momentum.RSIIndicator(close, window=72).rsi()
    feat['rsi_168h']   = ta.momentum.RSIIndicator(close, window=168).rsi()
    feat['macd_diff'] = ta.trend.MACD(close, window_slow=168, window_fast=72, window_sign=48
                                      ).macd_diff()
    feat['cci_168h'] = ta.trend.CCIIndicator(high, low, close, window=168).cci()

    # Stochastic Oscillator
    stoch = ta.momentum.StochasticOscillator(high, low, close, window=72, smooth_window=6)
    feat['stoch_k'] = stoch.stoch()
    feat['stoch_d'] = stoch.stoch_signal()

    # Williams %R
    feat['williams_r'] = ta.momentum.WilliamsRIndicator(high, low, close, lbp=72).williams_r()

    # EMA crossovers
    for w in [24, 72, 168]:
        ema = close.ewm(span=w, adjust=False).mean()
        feat[f'ema_{w}_ratio'] = (close / ema) - 1

    # ADX (Average Directional Index)
    feat['adx'] = ta.trend.ADXIndicator(high, low, close, window=120).adx()

    # ADVANCED VOLATILITY ESTIMATORS
    # Normalized ATR
    feat['atr_120h_pct'] = ta.volatility.AverageTrueRange(high, low, close, window=120).average_true_range() / close

    # Parkinson Volatility Estimator (uses High-Low range)
    feat['parkinson_vol_120h'] = np.sqrt(
        (1 / (4 * np.log(2))) * np.log(high / low) ** 2
    ).rolling(120).mean()

    # Garman-Klass Volatility
    gk = 0.5 * np.log(high / low) ** 2 - (2 * np.log(2) - 1) * np.log(close / opn) ** 2
    feat['gk_vol_120'] = np.sqrt(gk.rolling(120).mean())
    feat['gk_vol_336'] = np.sqrt(gk.rolling(336).mean())

    # Rolling realized volatility (annualized for daily data)
    feat['realvol_120h'] = log_ret.rolling(120).std() * np.sqrt(8760)

    # MICROSTRUCTURE & VOLUME
    feat['vol_ma_ratio']   = vol / (vol.rolling(168).mean() + 1e-8)
    feat['cmf'] = ta.volume.ChaikinMoneyFlowIndicator(high, low, close, vol, window=168).chaikin_money_flow()

    # HURST EXPONENT (Memory & Regime)
    def hurst_exponent(ts, min_lag=2, max_lag=48):
        """Estimate Hurst exponent via R/S analysis."""
        lags = range(min_lag, max_lag)
        tau  = [np.sqrt(np.std(np.subtract(ts[lag:], ts[:-lag]))) for lag in lags]
        if len(tau) < 2 or np.std(tau) == 0:
            return 0.5
        poly = np.polyfit(np.log(lags), np.log(tau), 1)
        return poly[0]

    feat['hurst'] = log_ret.fillna(0).rolling(168).apply(lambda x: hurst_exponent(x), raw=True)

    # FRACTIONAL DIFFERENTIATION (Stationarity + Memory)
    def get_weights_ffd(d: float, thresh: float = 1e-5):
        w, k = [1.0], 1
        while abs(w[-1]) >= thresh:
            w_ = -w[-1] / k * (d - k + 1)
            w.append(w_)
            k += 1
        return np.array(w[::-1])

    def frac_diff_ffd(series: pd.Series, d: float, thresh: float = 1e-5) -> pd.Series:
        w = get_weights_ffd(d, thresh)
        width = len(w) - 1
        output = []
        seriesF = series.fillna(method='ffill').dropna()
        for iloc in range(width, len(seriesF)):
            loc0 = seriesF.index[iloc - width]
            loc1 = seriesF.index[iloc]
            val  = np.dot(w, seriesF.loc[loc0:loc1])
            output.append((loc1, val))
        if not output:
            return pd.Series(dtype=float)
        idx, vals = zip(*output)
        return pd.Series(vals, index=idx)

    # d=0.4 is empirically robust for financial time series
    fd_series = frac_diff_ffd(np.log(close), d=0.4, thresh=1e-2)
    feat['frac_diff_04'] = fd_series.reindex(feat.index)

    # Bollinger Bands
    bb = ta.volatility.BollingerBands(close, window=168, window_dev=2)
    feat['bb_width_168h']  = (bb.bollinger_hband() - bb.bollinger_lband()) / bb.bollinger_mavg()
    feat['bb_pct_168h']    = bb.bollinger_pband()

    # Ichimoku Cloud
    tenkan  = (high.rolling(24).max()  + low.rolling(24).min())  / 2
    kijun   = (high.rolling(168).max() + low.rolling(168).min()) / 2
    senkou_a = ((tenkan + kijun) / 2).shift(168)
    senkou_b = ((high.rolling(336).max() + low.rolling(336).min()) / 2).shift(168)

    feat['ichi_above_cloud'] = ((close > senkou_a) & (close > senkou_b)).astype(int)
    feat['ichi_tk_diff']     = (tenkan - kijun) / close

    #Serial Correlation & Lags
    for lag in [1, 3, 12, 24, 168]:
        feat[f'ret_lag_{lag}h'] = log_ret.shift(lag)
        feat[f'acf_lag_{lag}h'] = log_ret.rolling(336).apply(
            lambda x: pd.Series(x).autocorr(lag=lag), raw=False
        )

    # Calendar Features
    feat['day_of_week']  = feat.index.dayofweek
    feat['hour_of_day']  = feat.index.hour
    feat['day_of_month'] = feat.index.day

    # CLEAN UP BEFORE HMM
    feat = feat.replace([np.inf, -np.inf], np.nan)
    feat = feat.iloc[336:].fillna(method='ffill').fillna(0)

    print(f"Feature matrix successfully built. Shape: {feat.shape}")
    return feat

features_all = compute_features(prices)

print(f'Feature matrix shape : {features_all.shape}')
print(f'Feature names ({len(features_all.columns)}):')
print(list(features_all.columns))

## Original cell 17 (zero-based index)

In [ ]:
print("Training Gaussian HMM...")

# Feature selection
hmm_features_all = features_all[['frac_diff_04', 'gk_vol_336']].copy()

hmm_features_all['ret_24h'] = np.log(prices['close'] / prices['close'].shift(24))
hmm_features_all.dropna(inplace=True)

# ROLLING SCALER
window_size = 720

# Calculate rolling mean and standard deviation
rolling_mean = hmm_features_all[['frac_diff_04', 'gk_vol_336']].rolling(window=window_size).mean()
rolling_std  = hmm_features_all[['frac_diff_04', 'gk_vol_336']].rolling(window=window_size).std()

# Dynamic Z-Score normalization
scaled_features = (hmm_features_all[['frac_diff_04', 'gk_vol_336']] - rolling_mean) / rolling_std
scaled_features.dropna(inplace=True) # Remove the first 720 rows which are NaN due to the rolling window

# Align hmm_features_all with the newly scaled data
hmm_features_all = hmm_features_all.loc[scaled_features.index]

# Train/Test split for the HMM
# Train only on the past so the model learns the transition matrices
train_mask = (scaled_features.index >= TRAIN_START) & (scaled_features.index <= TRAIN_END)
scaled_train = scaled_features.loc[train_mask]

# Fit HMM with 3 hidden states on training data
N_STATES = 3
RANDOM_STATE = 42

model_hmm = GaussianHMM(
    n_components=N_STATES,
    covariance_type='full',
    n_iter=200,
    random_state=RANDOM_STATE
)

print("Fitting HMM...")
model_hmm.fit(scaled_train.values)

# Prediction on all data
hmm_states = model_hmm.predict(scaled_features.values)
hmm_state_series = pd.Series(hmm_states, index=scaled_features.index, name='hmm_regime')

# labeling based on your 24h return
state_means = {}
for s in range(N_STATES):
    mask = hmm_state_series == s
    state_means[s] = hmm_features_all.loc[mask & train_mask, 'ret_24h'].mean()

sorted_states = sorted(state_means, key=state_means.get)
state_map = {old: new for new, old in enumerate(sorted_states)}
hmm_state_series = hmm_state_series.map(state_map)

# Update features_all
features_all['hmm_regime']     = hmm_state_series.reindex(features_all.index).fillna(1)
features_all['hmm_is_bull']    = (features_all['hmm_regime'] == 2).astype(int)
features_all['hmm_is_bear']    = (features_all['hmm_regime'] == 0).astype(int)
features_all['hmm_is_neutral'] = (features_all['hmm_regime'] == 1).astype(int)

print(f'\nHMM regime counts (0=Bear, 1=Neutral, 2=Bull):')
print(hmm_state_series.value_counts().sort_index())

# --- VISUALIZATION ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7), sharex=True)

colors_regime = {0: 'salmon', 1: 'lightgray', 2: 'lightgreen'}
labels_regime = {0: 'Bear', 1: 'Neutral', 2: 'Bull'}

ax1.plot(prices.index, prices['close'], color='steelblue', lw=1.2)
prev_state = hmm_state_series.iloc[0]
start_idx  = hmm_state_series.index[0]

for i, (dt, state) in enumerate(hmm_state_series.items()):
    if state != prev_state or i == len(hmm_state_series) - 1:
        ax1.axvspan(start_idx, dt,
                    color=colors_regime[prev_state], alpha=0.3,
                    label=labels_regime[prev_state])
        start_idx  = dt
        prev_state = state

handles, labels = ax1.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax1.legend(by_label.values(), by_label.keys(), loc='upper left')
ax1.set_title('BTC Price with HMM Regimes (Rolling Scaler + 24h Return)')
ax1.set_ylabel('Price (USD)')

ax2.plot(hmm_state_series.index, hmm_state_series, drawstyle='steps-post',
         color='purple', lw=0.8)
ax2.set_yticks([0, 1, 2])
ax2.set_yticklabels(['Bear', 'Neutral', 'Bull'])
ax2.set_title('HMM Hidden State (Regime)')

plt.tight_layout()
plt.show()

## Original cell 19 (zero-based index)

In [ ]:
print("Analyzing performance by HMM regime...")

# Create a temporary DataFrame for analysis
df_val = pd.DataFrame(index=features_all.index)
df_val['regime'] = features_all['hmm_regime'].map({0: '0_Bear', 1: '1_Neutral', 2: '2_Bull'})

# Calculate the 24h FORWARD return (What happens AFTER the signal is generated)
df_val['fwd_ret_24h'] = (prices['close'].shift(-24) / prices['close']) - 1

# Retrieve volatility (at the time of the signal)
df_val['volatility'] = features_all['gk_vol_336']

# Drop NaNs (introduced at the end of the dataset due to the forward shift)
df_val.dropna(inplace=True)

# Group by regime to compute statistical profiles
regime_stats = df_val.groupby('regime').agg(
    Observation_Count=('fwd_ret_24h', 'count'),
    Avg_Fwd_Return_24h_Pct=('fwd_ret_24h', lambda x: x.mean() * 100),
    Win_Rate_24h_Pct=('fwd_ret_24h', lambda x: (x > 0).mean() * 100),
    Avg_Volatility=('volatility', 'mean')
).round(2)

print("\n--- STATISTICAL PROFILE OF HMM REGIMES --- ")
display(regime_stats)

# Plotting the distribution of forward returns by regime
plt.figure(figsize=(10, 5))
sns.boxplot(
    x='regime',
    y='fwd_ret_24h',
    data=df_val,
    order=['0_Bear', '1_Neutral', '2_Bull'],
    palette=['salmon', 'lightgray', 'lightgreen']
)
plt.axhline(0, color='black', linestyle='--', lw=1)
plt.title('Distribution of Forward Returns (24h) by HMM Regime', fontsize=12, fontweight='bold')
plt.ylabel('24h Forward Return')
plt.xlabel('HMM Regime')

# Limit Y-axis to avoid distortion from extreme outliers
plt.ylim(-0.15, 0.15)
plt.tight_layout()
plt.show()

## Original cell 20 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Feature distribution summary & correlation heatmap
# ─────────────────────────────────────────────────────────────
print(f'Total features constructed : {len(features_all.columns)}')
print(f'NaN check: {features_all.isna().sum().sum()} NaN values')
print()
print('Feature statistics:')
display(features_all.describe().T.round(4))


# Compute the correlation matrix
corr_matrix = features_all.corr()
plt.figure(figsize=(18, 14))
sns.heatmap(corr_matrix,
            cmap='coolwarm',
            vmax=1.0,
            vmin=-1.0,
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": .6})

plt.title('Feature Correlation Matrix', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

## Original cell 22 (zero-based index)

In [ ]:
def _scan_single_t(i, t, log_prices_array, index_array, look_forward_min, look_forward_max):
    """Worker function for parallel processing."""
    n_total = len(log_prices_array)

    # Calculate how far forward we can look
    max_L_possible = min(look_forward_max, n_total - i - 1)

    if max_L_possible < look_forward_min:
        return None  # Not enough data forward

    best_tval = -np.inf
    best_sign = 1
    best_L    = look_forward_min
    best_tend = None

    for L in range(look_forward_min, max_L_possible + 1):
        y = log_prices_array[i : i + L + 1]
        x = np.arange(len(y))

        if len(x) < 3:
            continue

        # OLS using numpy polyfit for speed
        slope, intercept = np.polyfit(x, y, 1)
        y_hat = slope * x + intercept
        residuals = y - y_hat
        sse = np.sum(residuals ** 2)
        n = len(y)
        se = np.sqrt(sse / max(n - 2, 1) / np.sum((x - x.mean()) ** 2))

        tval = abs(slope / (se + 1e-10))

        if tval > best_tval:
            best_tval = tval
            best_sign = np.sign(slope)
            best_L    = L
            best_tend = index_array[i + L]

    return {
        'date':    t,
        'label':   int(best_sign),
        't_value': best_tval,
        'horizon': best_L,
        't_end':   best_tend
    }

def trend_scanning_labels_parallel(
    prices: pd.Series,
    look_forward_min: int = 5,
    look_forward_max: int = 21,
    n_jobs: int = -1
) -> pd.DataFrame:

    log_prices = np.log(prices).values
    index_array = prices.index.values
    t_events = prices.index

    print(f'Computing Trend Scanning Labels in parallel (n_jobs={n_jobs})...')

    results = Parallel(n_jobs=n_jobs)(
        delayed(_scan_single_t)(i, t, log_prices, index_array, look_forward_min, look_forward_max)
        for i, t in enumerate(t_events)
    )

    # Remove None values (from the end of the series where forward look isn't possible)
    results = [r for r in results if r is not None]

    df_labels = pd.DataFrame(results).set_index('date')
    df_labels['label'] = df_labels['label'].replace(0, 1)
    df_labels['label'] = df_labels['label'].apply(lambda x: 1 if x >= 0 else -1)

    # Clipping
    tValueVariance = df_labels['t_value'].var()
    tMax = min(30, tValueVariance)
    df_labels.loc[df_labels['t_value'] > tMax, 't_value'] = tMax
    df_labels.loc[df_labels['t_value'] < -tMax, 't_value'] = -tMax

    return df_labels

## Original cell 23 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Visualise labels on price chart
# ─────────────────────────────────────────────────────────────

labels_all = trend_scanning_labels_parallel(prices['close'], look_forward_min=12, look_forward_max=72)

print(f"Labels created with : {len(labels_all)} raws.")

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

# Panel 1: Price with colour-coded labels
bull_mask = labels_all['label'] == 1
bear_mask = labels_all['label'] == -1

axes[0].plot(prices.index, prices['close'], color='steelblue', lw=1, alpha=0.7)
axes[0].scatter(
    labels_all.index[bull_mask], prices.loc[labels_all.index[bull_mask], 'close'],
    c='green', s=0.5, alpha=0.5, label='Bull (+1)', zorder=3
)
axes[0].scatter(
    labels_all.index[bear_mask], prices.loc[labels_all.index[bear_mask], 'close'],
    c='red', s=0.5, alpha=0.5, label='Bear (-1)', zorder=3
)
axes[0].axvline(pd.Timestamp(TEST_START), color='black', linestyle='--',
                lw=1.5, label='Train/Test split')
axes[0].set_title('Trend Scanning Labels on BTC Price')
axes[0].set_ylabel('Close Price (USD)')
axes[0].legend(fontsize=9)

# Panel 2: t-values (confidence of label)
axes[1].fill_between(labels_all.index,
                      labels_all['t_value'] * labels_all['label'],
                      0,
                      where=(labels_all['label'] == 1),
                      color='green', alpha=0.5, label='Bull t-stat')
axes[1].fill_between(labels_all.index,
                      labels_all['t_value'] * labels_all['label'],
                      0,
                      where=(labels_all['label'] == -1),
                      color='red', alpha=0.5, label='Bear t-stat')
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_title('Signed t-Statistic of Optimal Trend')
axes[1].set_ylabel('Signed t-value')
axes[1].legend(fontsize=9)

# Panel 3: Optimal horizon
axes[2].bar(labels_all.index, labels_all['horizon'],
            color='purple', alpha=0.4, width=1)
axes[2].set_title('Optimal Look-Forward Horizon (hours)')
axes[2].set_ylabel('Horizon (hours)')

plt.tight_layout()
plt.show()


print(f"Average Optimal Horizon: {labels_all['horizon'].mean():.2f} hours")

## Original cell 25 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Align features and labels
# ─────────────────────────────────────────────────────────────
common_idx = features_all.index.intersection(labels_all.index)
X_all = features_all.loc[common_idx].copy()
y_all = labels_all.loc[common_idx, 'label'].copy()

embargo_hours = 120
TEST_START_EMBARGO = str(pd.to_datetime(TRAIN_END) + pd.Timedelta(hours=embargo_hours))

# Train / Test split
X_train = X_all.loc[TRAIN_START:TRAIN_END]
y_train = y_all.loc[TRAIN_START:TRAIN_END]
X_test  = X_all.loc[TEST_START_EMBARGO:TEST_END]
y_test  = y_all.loc[TEST_START_EMBARGO:TEST_END]

print(f'X_train shape : {X_train.shape} | y_train: {y_train.shape}')
print(f'X_test  shape : {X_test.shape}  | y_test:  {y_test.shape}')
print(f'\nClass balance (train): +1={( y_train==1).mean():.2%} | -1={(y_train==-1).mean():.2%}')
print(f'Class balance (test) : +1={( y_test==1).mean():.2%} | -1={(y_test==-1).mean():.2%}')

## Original cell 29 (zero-based index)

In [ ]:
class PurgedKFold:
    """
    Purged K-Fold Cross-Validation for financial time series.

    Prevents information leakage by:
    1. Purging: removing train samples whose label spans overlap with
       the test fold
    2. Embargo: removing additional samples immediately after the test
       fold

    Reference: López de Prado (2018), Chapter 7.
    """

    def __init__(self, n_splits: int = 5, embargo_pct: float = 0.01):
        self.n_splits    = n_splits
        self.embargo_pct = embargo_pct

    def split(self, X: pd.DataFrame, y=None, groups=None):
        n          = len(X)
        fold_size  = n // self.n_splits
        embargo_n  = int(n * self.embargo_pct)

        for k in range(self.n_splits):
            test_start  = k * fold_size
            test_end    = (k + 1) * fold_size if k < self.n_splits - 1 else n
            test_idx    = np.arange(test_start, test_end)

            # Embargo: remove samples just after test fold
            embargo_idx = np.arange(test_end, min(test_end + embargo_n, n))

            # Purge: remove samples just before test fold
            purge_idx   = np.arange(max(test_start - embargo_n, 0), test_start)

            excluded    = set(test_idx) | set(embargo_idx) | set(purge_idx)
            train_idx   = np.array([i for i in range(n) if i not in excluded])

            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


pkf = PurgedKFold(n_splits=5, embargo_pct=0.02)
print('PurgedKFold cross-validator initialized ✓')
print('Fold sizes:')
for i, (tr, te) in enumerate(pkf.split(X_train)):
    print(f'  Fold {i+1}: train={len(tr)}, test={len(te)}')

## Original cell 31 (zero-based index)

In [ ]:

cols_to_drop = ['day_of_month', 'day_of_week', 'hour_of_day']

X_train = X_train.drop(columns=cols_to_drop, errors='ignore')
X_test  = X_test.drop(columns=cols_to_drop, errors='ignore')

## Original cell 33 (zero-based index)

In [ ]:
# Convert labels {-1,+1} → {0,1} for sklearn
y_train_bin = ((y_train + 1) // 2).astype(int)   # -1→0, +1→1
y_test_bin  = ((y_test  + 1) // 2).astype(int)

# Scale features
scaler   = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Convert back to DataFrame (important for feature names)
X_train_s = pd.DataFrame(X_train_s, columns=X_train.columns, index=X_train.index)
X_test_s  = pd.DataFrame(X_test_s,  columns=X_test.columns,  index=X_test.index)

# Random Forest with sensible defaults
rf_base = RandomForestClassifier(
    n_estimators     = 500,
    max_depth        = 6,
    max_features     = 'sqrt',
    min_samples_leaf = 5,
    class_weight     = 'balanced',
    random_state     = RANDOM_STATE,
    n_jobs           = -1
)

# Purged K-Fold CV evaluation
cv_scores_rf = []
for fold, (tr_idx, te_idx) in enumerate(pkf.split(X_train_s)):
    X_cv_tr = X_train_s.iloc[tr_idx]
    y_cv_tr = y_train_bin.iloc[tr_idx]
    X_cv_te = X_train_s.iloc[te_idx]
    y_cv_te = y_train_bin.iloc[te_idx]

    rf_base.fit(X_cv_tr, y_cv_tr)
    preds  = rf_base.predict(X_cv_te)
    acc    = accuracy_score(y_cv_te, preds)
    cv_scores_rf.append(acc)
    print(f'  Fold {fold+1}: accuracy = {acc:.4f}')

print(f'\nRF Purged CV accuracy: {np.mean(cv_scores_rf):.4f} ± {np.std(cv_scores_rf):.4f}')

# Retrain on full training set
rf_base.fit(X_train_s, y_train_bin)
y_pred_rf  = rf_base.predict(X_test_s)
y_prob_rf  = rf_base.predict_proba(X_test_s)[:, 1]

print(f'\nRF Test accuracy  : {accuracy_score(y_test_bin, y_pred_rf):.4f}')
print(f'RF Test F1 (macro): {f1_score(y_test_bin, y_pred_rf, average="macro"):.4f}')

## Original cell 35 (zero-based index)

In [ ]:
def objective_xgb(trial):
    params = {
        'max_depth'          : trial.suggest_int('max_depth', 3, 7),
        'learning_rate'      : trial.suggest_float('learning_rate', 0.001, 0.05, log=True),
        'subsample'          : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'   : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight'   : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'          : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'         : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),

        'n_estimators'       : 2000,
        'early_stopping_rounds': 50,
        'use_label_encoder'  : False,
        'eval_metric'        : 'logloss',
        'random_state'       : RANDOM_STATE,
        'n_jobs'             : -1
    }

    fold_f1_scores = []
    fold_best_iters = []

    for tr_idx, te_idx in pkf.split(X_train_s):
        X_cv_tr = X_train_s.iloc[tr_idx].values
        y_cv_tr = y_train_bin.iloc[tr_idx].values
        X_cv_te = X_train_s.iloc[te_idx].values
        y_cv_te = y_train_bin.iloc[te_idx].values

        # Dynamic ratio
        current_ratio = (y_cv_tr == 0).sum() / (y_cv_tr == 1).sum()
        params['scale_pos_weight'] = current_ratio

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_cv_tr, y_cv_tr,
                eval_set=[(X_cv_te, y_cv_te)],
                verbose=False)

        # Retrieve the iteration where the model stopped (the optimal number of trees)
        fold_best_iters.append(clf.best_iteration)

        preds = clf.predict(X_cv_te)
        score = f1_score(y_cv_te, preds, average='macro')
        fold_f1_scores.append(score)

    # Calculate the average number of trees across the folds
    avg_best_iter = int(np.mean(fold_best_iters))

    # Save this information in Optuna's memory for this specific "trial"
    trial.set_user_attr("optimal_n_estimators", avg_best_iter)

    return np.mean(fold_f1_scores)

print('Running Optuna Bayesian search on F1-Score (100 trials)...')
study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective_xgb, n_trials=100, show_progress_bar=True)

print(f'\nBest CV F1-Score (Macro) : {study.best_value:.4f}')
print(f'Best parameters  :')
for k, v in study.best_params.items():
    print(f'  {k:25s}: {v}')

# Extract the ideal number of trees from the BEST trial
best_n_estimators = study.best_trial.user_attrs["optimal_n_estimators"]
print(f' Optimal number of trees found by Early Stopping : {best_n_estimators}')

## Original cell 37 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Train final XGBoost model on full training set
# ─────────────────────────────────────────────────────────────

final_ratio = (y_train_bin == 0).sum() / (y_train_bin == 1).sum()

best_xgb_params = {
    **study.best_params,
    'scale_pos_weight'   : final_ratio,
    'n_estimators'       : best_n_estimators,
    'use_label_encoder': False,
    'eval_metric'      : 'logloss',
    'random_state'     : RANDOM_STATE,
    'n_jobs'           : -1
}

xgb_model = xgb.XGBClassifier(**best_xgb_params)
xgb_model.fit(
    X_train_s, y_train_bin,
    verbose=False
)

y_pred_xgb  = xgb_model.predict(X_test_s)
y_prob_xgb  = xgb_model.predict_proba(X_test_s)[:, 1]

print('XGBoost final model trained')
print(f'Test accuracy  : {accuracy_score(y_test_bin, y_pred_xgb):.4f}')
print(f'Test F1 (macro): {f1_score(y_test_bin, y_pred_xgb, average="macro"):.4f}')
print(f'Test ROC-AUC   : {roc_auc_score(y_test_bin, y_prob_xgb):.4f}')

## Original cell 38 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# PREDICTION BIAS ANALYSIS
# ─────────────────────────────────────────────────────────────
print("--- MODEL DIRECTIONAL BIAS DIAGNOSTIC --- ")

# Calculate Distributions
actual_dist = pd.Series(y_test_bin).value_counts(normalize=True)
pred_dist   = pd.Series(y_pred_xgb).value_counts(normalize=True)

print("CLASS DISTRIBUTION COMPARISON")
print(f"   Actual Test Data : Bull={actual_dist.get(1,0):.1%} | Bear={actual_dist.get(0,0):.1%}")
print(f"   Model Predicted  : Bull={pred_dist.get(1,0):.1%} | Bear={pred_dist.get(0,0):.1%}")

# Bias Calculation
bias_gap = abs(pred_dist.get(1,0) - actual_dist.get(1,0))

print(f" BIAS METRIC")
print(f"   Directional Bias Gap: {bias_gap:.1%}")


## Original cell 41 (zero-based index)

In [ ]:
print(" TRAINING THE RISK MANAGER (RANDOM FOREST) ---")
X_meta_train = X_train_s.copy()
X_meta_train['primary_prob'] = primary_oof_probs

X_meta_test = X_test_s.copy()
X_meta_test['primary_prob'] = y_prob_xgb

meta_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=4,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
meta_model.fit(X_meta_train, meta_y_train)

print(" OPTIMIZING CONFIDENCE THRESHOLD (STRICTLY ON TRAIN SET) ---")
meta_oof_probs = cross_val_predict(
    meta_model, X_meta_train, meta_y_train,
    cv=pkf, method='predict_proba', n_jobs=-1
)[:, 1]


print(" INFERENCE ON 2021 (ONE-SHOT FILTERING) ---")
meta_test_probs = meta_model.predict_proba(X_meta_test)[:, 1]

meta_test_preds_opt = (meta_test_probs >= 0.45).astype(int)

filtered_preds = y_pred_xgb.copy()
filtered_preds[meta_test_preds_opt == 0] = 0 # 0 = Cash

total_initial_trades = len(y_pred_xgb)
total_filtered_trades = np.sum(meta_test_preds_opt == 1)
trades_cancelled = total_initial_trades - total_filtered_trades

print(f"\n--- TRADING IMPACT ---")
print(f"Initial trades proposed by Primary Model : {total_initial_trades}")
print(f"Trades cancelled by Risk Manager         : {trades_cancelled} ({(trades_cancelled/total_initial_trades)*100:.1f}%)")
print(f"Final trades executed                    : {total_filtered_trades}")

# FINAL METRICS
indices_kept = (meta_test_preds_opt == 1)

if np.sum(indices_kept) > 0:
    acc_before = accuracy_score(y_test_bin, y_pred_xgb)
    f1_before  = f1_score(y_test_bin, y_pred_xgb, average='macro')

    acc_after = accuracy_score(y_test_bin.values[indices_kept], y_pred_xgb[indices_kept])
    f1_after  = f1_score(y_test_bin.values[indices_kept], y_pred_xgb[indices_kept], average='macro')

    print(f"\n--- PERFORMANCE UPGRADE ---")
    print(f"BEFORE Filtering : Accuracy = {acc_before:.4f} | F1-Macro = {f1_before:.4f}")
    print(f"AFTER Filtering  : Accuracy = {acc_after:.4f} | F1-Macro = {f1_after:.4f}")
else:
    print("\nThe Meta-Model cancelled absolutely all trades! Market conditions deemed too uncertain.")

## Original cell 45 (zero-based index)

In [ ]:
print("--- OPTUNA SEARCH: END-OF-PERIOD WALK-FORWARD ---")

# Simulation parameters
VAL_SIZE = 1000   # ~Last 2 months of 2020
CHUNK_SIZE = 168  # Advancing by 168h (1 week) for faster optimization
HORIZON_EMBARGO = 72 #match look_forward_max)

X_train_np = X_train_s.values
y_train_np = y_train_bin.values

# Index where our 'mock 2021' simulation begins
split_idx = len(X_train_np) - VAL_SIZE

def objective_wf(trial):
    params = {
        'max_depth'          : trial.suggest_int('max_depth', 3, 8),
        'learning_rate'      : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample'          : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'   : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight'   : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'          : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda'         : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'n_estimators'       : trial.suggest_int('n_estimators', 100, 800),

        'tree_method'        : 'hist',
        'use_label_encoder'  : False,
        'eval_metric'        : 'logloss',
        'random_state'       : RANDOM_STATE,
        'n_jobs'             : -1
    }

    all_preds = []
    all_trues = []

    # Mini-Expanding Window simulation loop on late 2020
    # Start at split_idx and step forward by CHUNK_SIZE
    for i in range(split_idx, len(X_train_np), CHUNK_SIZE):

        # Embargo cut the last 120 hours to prevent look-ahead bias
        limit_idx = i - HORIZON_EMBARGO

        if limit_idx <= 0:
            continue

        # Training data expands at each iteration
        X_train_chunk = X_train_np[:limit_idx]
        y_train_chunk = y_train_np[:limit_idx]

        # The small prediction window
        X_test_chunk = X_train_np[i : i + CHUNK_SIZE]
        y_test_chunk = y_train_np[i : i + CHUNK_SIZE]

        if len(X_test_chunk) == 0:
            break

        # Dynamic Ratio : Recalculate class weights based on available history
        ratio = (y_train_chunk == 0).sum() / (y_train_chunk == 1).sum()
        params['scale_pos_weight'] = ratio

        # Model training and prediction
        clf = xgb.XGBClassifier(**params)
        clf.fit(X_train_chunk, y_train_chunk, verbose=False)

        preds = clf.predict(X_test_chunk)

        # Store predictions
        all_preds.extend(preds)
        all_trues.extend(y_test_chunk)

    # Final score computed over the entire simulation period
    return f1_score(all_trues, all_preds, average='macro')

print(f"Simulating the final {VAL_SIZE} hours of 2020 (Step size: {CHUNK_SIZE}h)...")
print("Launching Optuna (30 trials)...")

study_wf = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
# 30 trials are sufficient as this simulation closely mimics real-world testing
study_wf.optimize(objective_wf, n_trials=30, show_progress_bar=True)

print(f'Best F1-Score (Walk-Forward): {study_wf.best_value:.4f}')
print(f'Best hyperparameters (calibrated for 2021):')
for k, v in study_wf.best_params.items():
    print(f'  {k:25s}: {v}')

## Original cell 46 (zero-based index)

In [ ]:
print(f'Running Optimized Expanding-Window predictions (Chunks of 48h)...')
start_time = time.time()

best_xgb_params = {
    **study_wf.best_params,
    'tree_method'        : 'hist',
    'use_label_encoder': False,
    'eval_metric'      : 'logloss',
    'random_state'     : RANDOM_STATE,
    'n_jobs'           : -1
}

RETRAIN_FREQ  = 48    # retrain every 48 hours
MIN_TRAIN_OBS = 200   # minimum training observations

X_full = pd.concat([X_train_s, X_test_s])
y_full_bin = pd.concat([y_train_bin, y_test_bin])

test_dates  = X_test_s.index

ew_preds_list = []
ew_probs_list = []
date_indexes  = []

for i in range(0, len(test_dates), RETRAIN_FREQ):

    chunk_dates = test_dates[i : i + RETRAIN_FREQ]
    current_split_date = chunk_dates[0]

    # Embargo
    embargo_date = current_split_date - pd.Timedelta(hours=HORIZON_EMBARGO)

    train_mask = X_full.index < embargo_date
    X_ew_tr = X_full.loc[train_mask]
    y_ew_tr = y_full_bin.loc[train_mask]

    if len(X_ew_tr) < MIN_TRAIN_OBS:
        ew_preds_list.extend([1] * len(chunk_dates))
        ew_probs_list.extend([0.5] * len(chunk_dates))
        date_indexes.extend(chunk_dates)
        continue

    # Dynamic ratio
    current_ratio = (y_ew_tr == 0).sum() / (y_ew_tr == 1).sum()

    # Add this fresh ratio to our parameters
    chunk_params = best_xgb_params.copy()
    chunk_params['scale_pos_weight'] = current_ratio

    # Retrain the model
    ew_model = xgb.XGBClassifier(**chunk_params)
    ew_model.fit(X_ew_tr, y_ew_tr, verbose=False)

    # Predict the 48h chunk
    X_chunk_to_predict = X_full.loc[chunk_dates]

    chunk_preds = ew_model.predict(X_chunk_to_predict)
    chunk_probs = ew_model.predict_proba(X_chunk_to_predict)[:, 1]

    ew_preds_list.extend(chunk_preds)
    ew_probs_list.extend(chunk_probs)
    date_indexes.extend(chunk_dates)

ew_preds = np.array(ew_preds_list)
ew_probs = np.array(ew_probs_list)

elapsed = time.time() - start_time
print(f'Done in {elapsed:.2f} seconds.')
print(f'Expanding-window predictions: {len(ew_preds)}')
print(f'Test accuracy (expanding)  : {accuracy_score(y_test_bin, ew_preds):.4f}')
print(f'Test F1 (expanding, macro) : {f1_score(y_test_bin, ew_preds, average="macro"):.4f}')

## Original cell 49 (zero-based index)

In [ ]:
def compute_xgboost_importance(model, feature_names):
    """Extract feature importances (Average Gain for XGBoost)."""
    importances = model.feature_importances_
    df_imp = pd.DataFrame({
        'feature'   : feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    return df_imp

# Compute Importance for the finalized XGBoost model
importance_df = compute_xgboost_importance(xgb_model, X_train_s.columns)

# Plot top 25 features
top_n = 25
fig, ax = plt.subplots(figsize=(10, 8))
top_imp = importance_df.head(top_n)

# Color gradient for visual appeal
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, top_n))
bars = ax.barh(range(top_n), top_imp['importance'], color=colors, edgecolor='white', height=0.8)

ax.set_yticks(range(top_n))
ax.set_yticklabels(top_imp['feature'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (Average Gain)')
ax.set_title(f'Top {top_n} Features — Importance by Gain (XGBoost)', fontsize=12, fontweight='bold')

# Add value labels at the end of the bars
for i, (bar, imp) in enumerate(zip(bars, top_imp['importance'])):
    ax.text(imp + 0.001, i, f'{imp:.4f}', va='center', fontsize=7)

plt.tight_layout()
plt.show()

print(f'Sum of top {top_n} importances: {top_imp["importance"].sum():.4f}')

## Original cell 51 (zero-based index)

In [ ]:
def cluster_features(X: pd.DataFrame, n_clusters: int = 8):
    """
    Cluster features by their Pearson correlation structure.

    Uses hierarchical (Ward) clustering on the distance matrix
    derived from feature correlations: dist = sqrt(0.5*(1-corr)).

    Returns
    -------
    pd.Series: mapping feature → cluster_id
    linkage matrix
    """
    corr_mat  = X.corr().fillna(0)
    # Convert correlation to distance
    dist_mat  = np.sqrt(np.clip(0.5 * (1 - corr_mat), 0, 1))
    dist_condensed = squareform(dist_mat.values, checks=False)
    link      = linkage(dist_condensed, method='ward')
    clusters  = fcluster(link, t=n_clusters, criterion='maxclust')
    cluster_series = pd.Series(clusters, index=X.columns, name='cluster')
    return cluster_series, link, corr_mat


N_CLUSTERS = 10
cluster_map, linkage_mat, corr_mat = cluster_features(X_train, n_clusters=N_CLUSTERS)

print(f'Feature clusters ({N_CLUSTERS} clusters):')
for c in sorted(cluster_map.unique()):
    members = cluster_map[cluster_map == c].index.tolist()
    print(f'  Cluster {c:2d} ({len(members):3d} features): {", ".join(members[:5])}{"..." if len(members)>5 else ""}')

## Original cell 52 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Visualise correlation matrix with cluster structure
# ─────────────────────────────────────────────────────────────
# Sort features by cluster
sorted_feats = cluster_map.sort_values().index
sorted_corr  = corr_mat.loc[sorted_feats, sorted_feats]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Correlation heatmap
mask = np.zeros_like(sorted_corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(
    sorted_corr, ax=axes[0], cmap='RdBu_r', vmin=-1, vmax=1,
    xticklabels=False, yticklabels=False,
    cbar_kws={'label': 'Pearson Correlation'}
)
axes[0].set_title('Feature Correlation Matrix\n(sorted by cluster)')

# Dendrogram
dendrogram(
    linkage_mat, ax=axes[1],
    labels=X_train_s.columns.tolist(),
    leaf_font_size=6, orientation='right',
    color_threshold=0.7 * max(linkage_mat[:, 2])
)
axes[1].set_title('Hierarchical Clustering Dendrogram')
axes[1].set_xlabel('Distance')

plt.tight_layout()
plt.show()

## Original cell 53 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Clustered MDI: sum importance within each cluster
# ─────────────────────────────────────────────────────────────

def clustered_importance(importance_df, cluster_map):
    """
    Aggregate individual feature importances (Gain) to the cluster level.
    This reveals the true predictive power of correlated feature families.
    """
    df = importance_df.set_index('feature').copy()
    df['cluster'] = cluster_map

    # Sum importances within each cluster
    cluster_imp = df.groupby('cluster')['importance'].sum().sort_values(ascending=False)

    # Identify the representative feature per cluster (highest individual importance)
    rep_feats = df.groupby('cluster')['importance'].idxmax()

    result = pd.DataFrame({
        'cluster_importance': cluster_imp,
        'top_feature'       : rep_feats
    })

    return result.sort_values('cluster_importance', ascending=False)

# Assuming 'importance_df' comes from the previous XGBoost Gain block
clustered_gain = clustered_importance(importance_df, cluster_map)

print('--- CLUSTERED IMPORTANCE (XGBOOST GAIN) ---')
display(clustered_gain)

# ─────────────────────────────────────────────────────────────
# VISUALIZATION
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

# Generate a visually appealing color palette
pal = sns.color_palette('tab10', len(clustered_gain))

ax.bar(
    range(len(clustered_gain)),
    clustered_gain['cluster_importance'],
    color=pal,
    edgecolor='white',
    width=0.7
)

ax.set_xticks(range(len(clustered_gain)))

# Format X-axis labels: Cluster ID + Truncated Top Feature Name
ax.set_xticklabels(
    [f'Cluster {c}\n({f[:15]}...)' for c, f in zip(clustered_gain.index, clustered_gain['top_feature'])],
    fontsize=9,
    rotation=45,
    ha='right'
)

ax.set_ylabel('Summed Importance (Average Gain)')
ax.set_title('Clustered Feature Importance\n(Mitigating the Substitution Effect)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## Original cell 55 (zero-based index)

In [ ]:
"""
PFI measures the drop in model performance when feature values are
randomly shuffled. A feature is important if shuffling it degrades
performance significantly on unseen data.

Key differences from XGBoost Gain:
  - Gain is computed during training (In-Sample, prone to overfitting).
  - PFI is computed on held-out data (Test set / Out-Of-Sample).
  - PFI reveals which features actually drive generalization.
"""

print('Computing Permutation Feature Importance (n_repeats=20)...')

pfi_result = permutation_importance(
    xgb_model, X_test_s, y_test_bin,
    n_repeats    = 20,
    random_state = RANDOM_STATE,
    n_jobs       = -1,
    scoring      = 'f1_macro'
)

pfi_df = pd.DataFrame({
    'feature'    : X_test_s.columns,
    'importance' : pfi_result.importances_mean,
    'std'        : pfi_result.importances_std
}).sort_values('importance', ascending=False)

# Plot top 25 PFI with error bars
top_n = 25
top_pfi = pfi_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.coolwarm_r(np.linspace(0, 1, top_n))

ax.barh(
    range(top_n), top_pfi['importance'], xerr=top_pfi['std'],
    color=colors, edgecolor='white', height=0.8,
    error_kw={'elinewidth': 1.2, 'capsize': 3}
)

ax.set_yticks(range(top_n))
ax.set_yticklabels(top_pfi['feature'], fontsize=9)
ax.invert_yaxis()
ax.axvline(0, color='black', lw=0.8, linestyle='--')

ax.set_xlabel('Drop in F1-Score Macro (PFI)')
ax.set_title(f'Top {top_n} Features — Permutation Feature Importance (±1 SD)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Clustered PFI
clustered_pfi = clustered_importance(pfi_df, cluster_map)

print('\n--- CLUSTERED PFI (Top Feature Families) ---')
display(clustered_pfi.head())

## Original cell 56 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Compare MDI vs PFI
# ─────────────────────────────────────────────────────────────
print("Comparing In-Sample (Gain) vs Out-Of-Sample (PFI) Feature Importance...")

compare = importance_df.set_index('feature').rename(columns={'importance': 'Gain'}).join(
    pfi_df.set_index('feature')[['importance']].rename(columns={'importance': 'PFI'})
).dropna()

# Normalize both to [0,1] for fair comparison
compare['Gain_norm'] = compare['Gain'] / compare['Gain'].max()
compare['PFI_norm'] = (compare['PFI'] - compare['PFI'].min()) / \
                      (compare['PFI'].max() - compare['PFI'].min() + 1e-10)

# Visualization
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(compare['Gain_norm'], compare['PFI_norm'], alpha=0.6, s=50, c='steelblue')

# Annotate outliers
# High PFI, low Gain -> Undervalued / Robust generalizers (Red)
for feat, row in compare.nlargest(8, 'PFI_norm').iterrows():
    ax.annotate(feat[:15], (row['Gain_norm'], row['PFI_norm']),
                textcoords='offset points', xytext=(5, 0), fontsize=8, color='darkred', fontweight='bold')

# High Gain, low PFI -> Spurious / Overfitted features (Orange)
for feat, row in compare.nlargest(5, 'Gain_norm').iterrows():
    if row['PFI_norm'] < 0.3:
        ax.annotate(feat[:15], (row['Gain_norm'], row['PFI_norm']),
                    textcoords='offset points', xytext=(5, -10), fontsize=8, color='darkorange', fontweight='bold')

corr = compare['Gain_norm'].corr(compare['PFI_norm'])

ax.set_xlabel('Normalized XGBoost Gain (In-Sample)')
ax.set_ylabel('Normalized PFI (Out-Of-Sample / F1-Macro Drop)')
ax.set_title(f'Feature Robustness Check: Gain vs PFI (Pearson r = {corr:.3f})', fontsize=12, fontweight='bold')

# Add 45-degree line (baseline for perfect agreement)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='45° Perfect Agreement Line')
ax.legend()

plt.tight_layout()
plt.show()

# Consensus ranking
print(f'Gain–PFI Pearson correlation: {corr:.4f}')
print('\nMost Robust Features (Top 10 by average rank):')

# Calculate average rank (lower rank is better)
compare['avg_rank'] = (
    compare['Gain_norm'].rank(ascending=False) +
    compare['PFI_norm'].rank(ascending=False)
) / 2

display(compare.nsmallest(10, 'avg_rank')[['Gain', 'PFI']].round(6))

## Original cell 58 (zero-based index)

In [ ]:
print("Calculating SHAP values (opening the 'Black Box')...")

# Initialize the SHAP explainer with our global XGBoost model
explainer = shap.TreeExplainer(xgb_model)

# To avoid lengthy computation times, we take a representative
# sample of 2000 random rows from our training data (Train Set)
X_shap_sample = X_train_s.sample(n=2000, random_state=RANDOM_STATE)

# Compute the SHAP values matrix
shap_values = explainer.shap_values(X_shap_sample)

# Display the SHAP Summary Plot
plt.figure(figsize=(12, 8))
plt.title("SHAP Summary Plot: How XGBoost makes its decisions", fontsize=14, fontweight='bold', pad=20)

# The summary_plot handles the creation of its own figure elements
shap.summary_plot(
    shap_values,
    X_shap_sample,
    max_display=20,  # Show the Top 20 most impactful features
    plot_type="dot", # Red/blue dots to visualize directionality
    show=False       # Keeps the plot open for further Matplotlib adjustments
)

plt.tight_layout()
plt.show()

## Original cell 61 (zero-based index)

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name):
    """
    Full evaluation report for a binary classifier.

    Returns a dict of metrics and prints a formatted table.
    """
    metrics = {
        'accuracy'        : accuracy_score(y_true, y_pred),
        'precision_macro' : precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro'    : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro'        : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_bull'  : precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall_bull'     : recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_bull'         : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision_bear'  : precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        'recall_bear'     : recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        'f1_bear'         : f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        'roc_auc'         : roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan,
    }
    print(f'\n{'='*55}')
    print(f'  {model_name}')
    print(f'{'='*55}')
    print(f'  Accuracy        : {metrics["accuracy"]:.4f}')
    print(f'  F1 (macro)      : {metrics["f1_macro"]:.4f}')
    print(f'  ROC-AUC         : {metrics["roc_auc"]:.4f}')
    print(f'  Precision (Bull): {metrics["precision_bull"]:.4f}')
    print(f'  Recall (Bull)   : {metrics["recall_bull"]:.4f}')
    print(f'  Precision (Bear): {metrics["precision_bear"]:.4f}')
    print(f'  Recall (Bear)   : {metrics["recall_bear"]:.4f}')
    print(f'  Classification Report:')
    print(classification_report(y_true, y_pred, target_names=['Bear', 'Bull']))
    return metrics


metrics_rf  = evaluate_model(y_test_bin, y_pred_rf,  y_prob_rf,  'Random Forest (Baseline)')
metrics_xgb = evaluate_model(y_test_bin, y_pred_xgb, y_prob_xgb, 'XGBoost (Tuned, One-Shot)')
metrics_ew  = evaluate_model(y_test_bin, ew_preds,   ew_probs,   'XGBoost (Expanding Window)')
metrics_meta = evaluate_model(y_test_bin[indices_kept], y_pred_xgb[indices_kept], y_prob_xgb[indices_kept], 'XGBoost + Meta-Labeling (Filter)')

## Original cell 62 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Confusion matrices
# ─────────────────────────────────────────────────────────────
print("Generating Confusion Matrices to analyze directional bias...")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
models = [
    ('Random Forest (Baseline)', y_pred_rf),
    ('XGBoost (One-Shot)',       y_pred_xgb),
    ('XGBoost (Expanding Win.)', ew_preds),
    ('XGBoost + Meta-Label', filtered_preds),
]

for ax, (name, preds) in zip(axes, models):
    cm   = confusion_matrix(y_test_bin, preds)

    # Added (0) and (1) to the labels for better clarity
    disp = ConfusionMatrixDisplay(cm, display_labels=['Bear (0)', 'Bull (1)'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(
        f'{name}\nAcc = {accuracy_score(y_test_bin, preds):.3f} | F1-Macro = {f1_score(y_test_bin, preds, average="macro"):.3f}',
        fontsize=10
    )

plt.suptitle('Confusion Matrices — 2021 Test Set (Evaluating Directional Bias)', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## Original cell 63 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# ROC Curves & Precision-Recall Curves
# ─────────────────────────────────────────────────────────────

print("Generating ROC and Precision-Recall curves...")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
model_results = [
    ('Random Forest (Baseline)', y_prob_rf,  'steelblue'),
    ('XGBoost (One-Shot)',       y_prob_xgb, 'darkorange'),
    ('XGBoost (Expanding Win.)', ew_probs,   'forestgreen'),
    ('XGB + Meta-Label', meta_test_probs, 'crimson'),
]

for name, probs, color in model_results:
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test_bin, probs)
    roc_auc     = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

    # Precision-Recall Curve
    prec, rec, _ = precision_recall_curve(y_test_bin, probs)
    pr_auc       = auc(rec, prec)
    axes[1].plot(rec, prec, color=color, lw=2, label=f'{name} (AUC = {pr_auc:.3f})')

# --- Formatting ROC Plot ---
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Guess (AUC = 0.500)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9, loc='lower right')

# --- Formatting Precision-Recall Plot ---
base_rate = y_test_bin.mean()
axes[1].axhline(
    base_rate, color='k', linestyle='--', lw=1,
    label=f'Random Baseline (Rate = {base_rate:.3f})'
)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

## Original cell 64 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Model predictions on test price chart
# ─────────────────────────────────────────────────────────────

print("Plotting trading signals and model confidence over 2021...")


test_close = prices.loc[X_test_s.index, 'close']
ew_signal  = pd.Series(ew_preds, index=X_test_s.index)  # 0 = Bear, 1 = Bull
ew_prob_series = pd.Series(ew_probs, index=X_test_s.index)

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

# --- Panel 1: Pure Price ---
axes[0].plot(test_close.index, test_close.values, color='steelblue', lw=1.5)
axes[0].set_title('BTC/USD Close Price — 2021 Test Period', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Price (USD)')

# --- Panel 2: Trading Signals Overlaid ---
bull_dates = ew_signal[ew_signal == 1].index
bear_dates = ew_signal[ew_signal == 0].index

# Plot underlying price in gray
axes[1].plot(test_close.index, test_close.values, color='gray', lw=1, alpha=0.5)

# Overlay signals
axes[1].scatter(bull_dates, test_close.loc[bull_dates], c='forestgreen', s=1,
                alpha=0.8, label='Model: Bull (+1)', zorder=3)
axes[1].scatter(bear_dates, test_close.loc[bear_dates], c='crimson', s=1,
                alpha=0.8, label='Model: Bear (-1)', zorder=3)

axes[1].set_title('XGBoost Expanding-Window Signals Overlay (2021)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Price (USD)')
axes[1].legend(fontsize=10, loc='upper left')

# --- Panel 3: Model Confidence (Probabilities) ---
axes[2].fill_between(
    ew_prob_series.index, 0.5, ew_prob_series,
    where=(ew_prob_series >= 0.5), color='forestgreen', alpha=0.4, label='Bullish Confidence'
)
axes[2].fill_between(
    ew_prob_series.index, 0.5, ew_prob_series,
    where=(ew_prob_series < 0.5), color='crimson', alpha=0.4, label='Bearish Confidence'
)
axes[2].axhline(0.5, color='black', linestyle='--', lw=1)

axes[2].set_title('Predicted Probability of Bull Signal / Conviction Strength', fontsize=12, fontweight='bold')
axes[2].set_ylabel('P(Bull)')
axes[2].legend(fontsize=10, loc='upper right')

plt.tight_layout()
plt.show()

## Original cell 65 (zero-based index)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Summary comparison table
# ─────────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {
        'Model'       : 'Random Forest',
        'Accuracy'    : metrics_rf['accuracy'],
        'F1 (macro)'  : metrics_rf['f1_macro'],
        'ROC-AUC'     : metrics_rf['roc_auc'],
        'Precision Bull': metrics_rf['precision_bull'],
        'Recall Bull' : metrics_rf['recall_bull'],
        'Precision Bear' : metrics_rf['precision_bear'],
        'Recall Bear'    : metrics_rf['recall_bear'],
    },
    {
        'Model'       : 'XGBoost (one-shot)',
        'Accuracy'    : metrics_xgb['accuracy'],
        'F1 (macro)'  : metrics_xgb['f1_macro'],
        'ROC-AUC'     : metrics_xgb['roc_auc'],
        'Precision Bull': metrics_xgb['precision_bull'],
        'Recall Bull' : metrics_xgb['recall_bull'],
        'Precision Bear' : metrics_xgb['precision_bear'],
        'Recall Bear'    : metrics_xgb['recall_bear'],
    },
    {
        'Model'       : 'XGBoost (expanding window)',
        'Accuracy'    : metrics_ew['accuracy'],
        'F1 (macro)'  : metrics_ew['f1_macro'],
        'ROC-AUC'     : metrics_ew['roc_auc'],
        'Precision Bull': metrics_ew['precision_bull'],
        'Recall Bull' : metrics_ew['recall_bull'],
        'Precision Bear' : metrics_ew['precision_bear'],
        'Recall Bear'    : metrics_ew['recall_bear'],
    },
    {
        'Model': 'XGBoost + Meta-Labeling',
        'Accuracy'    : metrics_meta['accuracy'],
        'F1 (macro)'  : metrics_meta['f1_macro'],
        'ROC-AUC'     : metrics_meta['roc_auc'],
        'Precision Bull': metrics_meta['precision_bull'],
        'Recall Bull' : metrics_meta['recall_bull'],
        'Precision Bear' : metrics_meta['precision_bear'],
        'Recall Bear'    : metrics_meta['recall_bear'],
    },
]).set_index('Model')

print('\nModel Comparison — 2021 Test Set')
display(summary.round(4).style
        .background_gradient(cmap='RdYlGn', axis=0)
        .format('{:.4f}'))

## Original cell 67 (zero-based index)

In [ ]:
print("Computing rigorous Backtest metrics...")

# ── 6.1 CORE CALCULATIONS (Preventing Look-ahead bias) ────────
# Convert predictions [0, 1] to positions [-1, 1] (Short/Long)
position_series = pd.Series(ew_preds, index=test_dates).map({0: -1, 1: 1})
market_returns = prices['close'].pct_change().loc[test_dates].fillna(0)

# Shift positions by 1 to simulate trading at the next open/close
# This guarantees absolutely zero look-ahead bias in the backtest.
strat_returns = position_series.shift(1) * market_returns
strat_returns = strat_returns.fillna(0)

# Hourly data requires 8760 periods for annualization
HOURS_IN_YEAR = 8760

def get_metrics(returns, risk_free_rate=0.0):
    volatility = returns.std() * np.sqrt(HOURS_IN_YEAR)
    sharpe = ((returns.mean() * HOURS_IN_YEAR) - risk_free_rate) / (volatility + 1e-8)

    downside_returns = returns[returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(HOURS_IN_YEAR)
    sortino = ((returns.mean() * HOURS_IN_YEAR) - risk_free_rate) / (downside_vol + 1e-8)

    cum_ret = (1 + returns).cumprod()
    peaks = cum_ret.cummax()
    max_dd = ((cum_ret - peaks) / peaks).min()

    total_ret = cum_ret.iloc[-1] - 1
    return total_ret, sharpe, sortino, max_dd, cum_ret

tr_s, sh_s, so_s, dd_s, cum_strat = get_metrics(strat_returns)
tr_bh, sh_bh, so_bh, dd_bh, cum_bh = get_metrics(market_returns)

# ── 6.2 BUILD THE 'bt' DICTIONARY FOR VISUALS ─────────────────
bt = {
    '_cum_strat': cum_strat, '_cum_bh': cum_bh,
    '_position': position_series, '_strat_ret': strat_returns,
    'total_ret_strat': tr_s, 'total_ret_bh': tr_bh,
    'sharpe_strat': sh_s, 'sharpe_bh': sh_bh,
    'sortino_strat': so_s, 'sortino_bh': so_bh,
    'max_dd_strat': dd_s, 'max_dd_bh': dd_bh
}

# ── 6.3 VISUALISATION: 4-PANEL TEAR SHEET ─────────────────────
fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1.5, 1.5, 1.5]})

# Panel 1: Cumulative wealth
axes[0].plot(cum_strat.index, cum_strat.values, color='darkgreen', lw=2,
             label=f'XGBoost Strategy (Total={bt["total_ret_strat"]:.1%})')
axes[0].plot(cum_bh.index, cum_bh.values, color='steelblue', lw=2, linestyle='--',
             label=f'Buy & Hold (Total={bt["total_ret_bh"]:.1%})')
axes[0].set_ylabel('Cumulative Return (×)')
axes[0].set_title('Strategy vs Buy-and-Hold — Cumulative Wealth (2021)')
axes[0].legend()
axes[0].axhline(1.0, color='gray', lw=0.8, linestyle=':')

rel_perf = cum_strat / cum_bh
axes[0].fill_between(rel_perf.index, cum_strat, cum_bh, where=(cum_strat >= cum_bh),
                     alpha=0.08, color='green', label='Outperforms')
axes[0].fill_between(rel_perf.index, cum_strat, cum_bh, where=(cum_strat < cum_bh),
                     alpha=0.08, color='red', label='Underperforms')

# Panel 2: Drawdown
def drawdown_series(cum_ret):
    roll_max = cum_ret.cummax()
    return (cum_ret - roll_max) / roll_max

dd_strat = drawdown_series(cum_strat)
dd_bh    = drawdown_series(cum_bh)

axes[1].fill_between(dd_strat.index, dd_strat.values, 0, color='darkgreen', alpha=0.4, label='Strategy')
axes[1].fill_between(dd_bh.index, dd_bh.values, 0, color='steelblue', alpha=0.2, label='Buy & Hold')
axes[1].set_ylabel('Drawdown')
axes[1].legend(fontsize=9)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Panel 3: Position
axes[2].fill_between(bt['_position'].index, 0, bt['_position'].values,
                     step='post', color='purple', alpha=0.5, label='Position')
axes[2].set_ylabel('Position')
axes[2].set_yticks([-1, 1])
axes[2].set_yticklabels(['Short (-1)', 'Long (+1)'])
axes[2].set_title('Model Position Allocation')

# Panel 4: Rolling Sharpe
roll_sharpe = strat_returns.rolling(720).apply(
    lambda x: np.sqrt(HOURS_IN_YEAR) * x.mean() / (x.std() + 1e-10)
)
axes[3].plot(roll_sharpe.index, roll_sharpe.values, color='orange', lw=1.5)
axes[3].axhline(0, color='gray', lw=0.8, linestyle='--')
axes[3].axhline(bt['sharpe_bh'], color='steelblue', lw=1.2, linestyle=':',
                label=f'B&H Sharpe ({bt["sharpe_bh"]:.2f})')
axes[3].set_ylabel('Rolling Sharpe')
axes[3].set_title('30-Day Rolling Sharpe Ratio')
axes[3].legend(fontsize=9)

plt.tight_layout()
plt.show()

# ── 6.4 MONTHLY HEATMAP ───────────────────────────────────────
def monthly_returns_matrix(ret_series):
    # (1+R) cumulative product minus 1
    monthly = (1 + ret_series).resample('ME').prod() - 1
    df = monthly.to_frame('ret')
    df['year']  = df.index.year
    df['month'] = df.index.month
    pivot = df.pivot(index='year', columns='month', values='ret')
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]
    return pivot

strat_monthly = monthly_returns_matrix(strat_returns)
bh_monthly    = monthly_returns_matrix(market_returns)

fig, axes = plt.subplots(2, 1, figsize=(13, 5))

def plot_monthly_heatmap(df, ax, title):
    sns.heatmap(df, ax=ax, annot=True, fmt='.1%', center=0, cmap='RdYlGn',
                linewidths=0.5, linecolor='white',
                cbar_kws={'format': PercentFormatter(1.0, decimals=0)})
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('')

plot_monthly_heatmap(strat_monthly, axes[0], 'Strategy Monthly Returns (2021)')
plot_monthly_heatmap(bh_monthly,    axes[1], 'Buy-and-Hold Monthly Returns (2021)')
plt.tight_layout()
plt.show()

# ── 6.5 EXECUTIVE SUMMARY ─────────────────────────────────────
acc = accuracy_score(y_test_bin, ew_preds)
f1 = f1_score(y_test_bin, ew_preds, average="macro")

print("""
╔══════════════════════════════════════════════════════════════╗
║           CRYPTO ML STRATEGY — EXECUTIVE SUMMARY             ║
╠══════════════════════════════════════════════════════════════╣
║  METHODOLOGY                                                 ║
║  • Feature set: {n_feats} features across categories         ║
║    (Fractional Diff, Garman-Klass, HMM, Microstructure)      ║
║  • Labels: Trend Scanning (López de Prado), binary {{-1, 1}} ║
║  • Model: Bayesian-Tuned XGBoost (Expanding Window, 48h)     ║
║  • Validation: Purged K-Fold CV + Embargo                    ║
╠══════════════════════════════════════════════════════════════╣
║  TEST PERFORMANCE (2021 OOS)                                 ║
║  Classification Accuracy : {acc:>7.2%}                       ║
║  F1 Score (macro)        : {f1:>7.4f}                        ║
╠══════════════════════════════════════════════════════════════╣
║  BACKTEST PERFORMANCE (2021)                                 ║
║                    Strategy    Buy & Hold                    ║
║  Total Return   :  {tr_s:>8.1%}    {tr_bh:>8.1%}             ║
║  Sharpe Ratio   :  {sh_s:>8.3f}    {sh_bh:>8.3f}             ║
║  Max Drawdown   :  {dd_s:>8.1%}    {dd_bh:>8.1%}             ║
║  Sortino Ratio  :  {so_s:>8.3f}    {so_bh:>8.3f}             ║
╚══════════════════════════════════════════════════════════════╝
""".format(
    n_feats = len(features_all.columns),
    acc     = acc,
    f1      = f1,
    tr_s    = bt['total_ret_strat'],
    tr_bh   = bt['total_ret_bh'],
    sh_s    = bt['sharpe_strat'],
    sh_bh   = bt['sharpe_bh'],
    dd_s    = bt['max_dd_strat'],
    dd_bh   = bt['max_dd_bh'],
    so_s    = bt['sortino_strat'],
    so_bh   = bt['sortino_bh'],
))